# Simplified Copy-Trade Backtest

Simulates copying trades of a hand-picked set of wallets against the
processed Polygon trade shards.

## Strategy
- When a watched wallet prints a **BUY** on token `(condition_id, outcome)` at price `p`
  we place a buy order at a **slightly worse** price: `order_price = clip(p + WORSE_PRICE_OFFSET, 0.001, 0.999)`.
- The order is matched against the **first subsequent trade** whose effective price is
  `<= order_price` within `FILL_HORIZON_SECONDS`.
  - *same-token* liquidity: later BUY trades on the same `(condition_id, outcome)`.
  - *opposite-token* liquidity: later SELL trades on the **complementary** outcome
    (binary markets only), with effective price = `1 - sell_price`.
- For a wallet **SELL** the mirror applies: order at `p - WORSE_PRICE_OFFSET`, filled
  by the first later trade with effective price `>= order_price`.
  - same-token: later SELL prints on the same outcome.
  - opposite-token: later BUY prints on the complementary outcome, effective price `1 - buy_price`.

## Sharding
Trades are partitioned by the first hex character of `condition_id` (after `0x`).
All shards are processed in parallel; within each shard the backtest is exact
because a `condition_id` always falls in exactly one shard file.

## Outputs
One event ledger DataFrame with columns:
- `row_type` — `trigger` (watched-wallet trade that generated an order) or `fill` (our fill).
- `order_id` — unique integer identifying the copy order.
- `wallet` — originating wallet (`fill` rows carry the trigger wallet for reference).
- `condition_id`, `outcome`, `side`, `dt`, `tx_hash`, `price`.
- `order_price` — the price our order was placed at (`NaN` for fill rows).
- `fill_source` — `same_token` / `opposite_token` / `NaN` for trigger rows.
- `token_winner` — market resolution flag.
- `fill_pnl_usdc` — PnL of *our* position on fill rows (NaN for trigger rows),
  computed as stake × ( 1/exec_price − 1 ) for winning tokens, −stake for losers.

## Visualisation
Cumulative PnL chart with two series:
- **Wallet cohort** — raw `trade_pnl` of the watched wallets.
- **Copy strategy** — `fill_pnl_usdc` of our simulated fills.


## Configuration

In [1]:
%load_ext autoreload
%autoreload 2

import datetime
from pathlib import Path

# ── Input wallets ─────────────────────────────────────────────────────────────
# Replace with any wallet addresses you want to copy.
# Example: load from a CSV, a workspace registry, or define inline.
WATCHED_WALLETS: list[str] = None

ONLY_BUY_TRIGGERS = False
MAX_EXPOSURE_PER_WALLET_1h = 100

# ── Test period ───────────────────────────────────────────────────────────────
# None = use entire dataset (start/end are inferred from the data).
# Set to datetime.date objects to restrict the window.
PERIOD_START: datetime.date | None = datetime.date(2026, 2, 16)
# PERIOD_START: datetime.date | None = datetime.date(2026, 1, 1) # None   # e.g. datetime.date(2026, 2, 16)
PERIOD_END:   datetime.date | None = datetime.date(2026, 4, 20)
# PERIOD_END:   datetime.date | None = datetime.date(2026, 2, 16) # None   # e.g. datetime.date(2026, 3, 11)

# ── Copy-trade execution parameters ──────────────────────────────────────────
WORSE_PRICE_OFFSET: float = 0
FILL_HORIZON_SECONDS: float = 300     # max seconds after trigger to search for a fill
STAKE_USDC: float = 10.0               # max USDC per order (order qty = min(trigger_qty, STAKE_USDC / order_price))
FEE_BPS: float = 10.0                   # taker fee in basis points
MAX_POSITION_PER_CONDITION: float | None = 1000  # max qty per (wallet, condition_id) across all outcomes; None = unlimited
MAX_POSITION_PER_WALLET:    float | None = 10  # max qty per wallet across all conditions; None = unlimited

# ── Data ─────────────────────────────────────────────────────────────────────
TRADES_DIR     = Path("../../data/polygon_trades_processed")
RAW_TRADES_DIR = Path("../../data/trades_polygon")

# ── Parallelism ───────────────────────────────────────────────────────────────
MAX_WORKERS: int = 10   # number of threads for parallel shard processing

In [2]:
banned_wallets = set()
if(WATCHED_WALLETS is not None):
    WATCHED_WALLETS = [w for w in WATCHED_WALLETS if w not in banned_wallets]

## Optionally load wallets from stage-1 workspace

If `WATCHED_WALLETS` is empty above, this cell loads the wallet set from the
stage-1 strategy registry. Skip or modify as needed.

In [3]:
import pandas as pd

if not WATCHED_WALLETS:
    WORKSPACE_DIR = Path("../../data/trade_signals_workspace_v2")
    wallets_path = WORKSPACE_DIR / "selected_wallets_v2.parquet"
    if wallets_path.exists():
        _wallets_df = pd.read_parquet(wallets_path, columns=["wallet"])
        WATCHED_WALLETS = _wallets_df["wallet"].tolist()
        print(f"Loaded {len(WATCHED_WALLETS)} wallets from {wallets_path.name}")
    else:
        print("No wallet source found — set WATCHED_WALLETS manually or run stage1 first.")
else:
    print(f"Using {len(WATCHED_WALLETS)} manually specified wallets.")

Loaded 208 wallets from selected_wallets_v2.parquet


## Discover shards and derive test period

In [4]:
import pandas as pd

SHARD_PATHS: list[Path] = sorted(TRADES_DIR.glob("*.parquet"))
print(f"Found {len(SHARD_PATHS)} shards under {TRADES_DIR}")

# Derive END_DATE_TRAIN from the is_train flag (last date where is_train=True).
# Test data is everything strictly after END_DATE_TRAIN.
_sample = pd.read_parquet(SHARD_PATHS[0], columns=["dt", "is_train"])
_sample["dt"] = pd.to_datetime(_sample["dt"], utc=True)
END_DATE_TRAIN: datetime.date = _sample.loc[_sample["is_train"], "dt"].max().date()
_data_end: datetime.date      = _sample["dt"].max().date()
del _sample
print(f"END_DATE_TRAIN (last train date) : {END_DATE_TRAIN}")

# Backtest always runs on test data only (strictly after END_DATE_TRAIN).
# PERIOD_START / PERIOD_END can narrow the window further, but cannot go
# earlier than the day after END_DATE_TRAIN.
#_test_start = END_DATE_TRAIN + datetime.timedelta(days=1)
period_start: datetime.date = PERIOD_START # datetime.date(2026, 2, 16) #  max(PERIOD_START, _test_start) if PERIOD_START is not None else _test_start
period_end:   datetime.date = PERIOD_END #if PERIOD_END is not None else _data_end
print(f"Backtest period : {period_start}  →  {period_end}")

Found 16 shards under ../../data/polygon_trades_processed
END_DATE_TRAIN (last train date) : 2026-02-15
Backtest period : 2026-02-16  →  2026-04-20


## Backtest engine (per-shard)

Each shard is processed independently:
1. Load only rows within the backtest period (test data only).
2. Identify trigger events (watched-wallet trades).
3. Build a per-`condition_id` opposite-outcome map (binary markets only).
4. Process triggers in time order, maintaining a copy-portfolio **position** per `(wallet, condition_id, outcome)`.
5. For each trigger, place a copy order:
   - **BUY**: `qty_ordered = min(trig_qty, STAKE_USDC / order_price)` — worst-case loss = `qty × order_price ≤ STAKE_USDC`
   - **SELL**: `qty_ordered = min(trig_qty, position, STAKE_USDC / (1 − order_price))` — worst-case loss = `qty × (1 − order_price) ≤ STAKE_USDC`. Skipped if position = 0 (no short selling).
6. Scan tape prints in time order within `FILL_HORIZON_SECONDS`. Each matching print partially fills the order: `fill_qty = min(remaining_qty, tape_qty)`.
7. One **fill** ledger row is emitted per partial fill. BUY fills increment position; SELL fills decrement it.

`fill_pnl_usdc` per fill row (all executed at `exec_price = order_price`, limit fill):
- **BUY winner**: `fill_qty × (1 − exec_price) − fill_qty × exec_price × fee`
- **BUY loser**:  `−fill_qty × exec_price × (1 + fee)`
- **SELL winner** (sold below par): `fill_qty × (exec_price − 1) − fill_qty × exec_price × fee`
- **SELL loser** (sold above par):  `fill_qty × exec_price × (1 − fee)`


In [5]:
import numpy as np
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

# ─────────────────────────────────────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────────────────────────────────────

def _build_complement_map(df: pd.DataFrame) -> dict[tuple[str, str], str]:
    """Return {(condition_id, outcome): opposite_outcome} for binary markets."""
    pairs: dict[str, set] = {}
    for cid, grp in df.groupby("condition_id", sort=False):
        pairs[cid] = set(grp["outcome"].dropna().unique())
    result: dict[tuple[str, str], str] = {}
    for cid, outcomes in pairs.items():
        if len(outcomes) == 2:
            a, b = sorted(outcomes)
            result[(cid, a)] = b
            result[(cid, b)] = a
    return result


def _simulate_shard(
    shard_path: Path,
    raw_tape_path: Path,
    watched_wallets: set[str],
    period_start: datetime.date,
    period_end: datetime.date,
    worse_price_offset: float,
    fill_horizon_seconds: float,
    stake_usdc: float,
    fee_bps: float,
    max_position_per_condition: float | None = None,
    max_position_per_wallet: float | None = None,
) -> pd.DataFrame:
    """Process one shard file and return an event ledger DataFrame.

    Triggers are read from ``shard_path`` (processed per-wallet aggregated trades),
    which supplies ``token_winner`` and ``trade_pnl``.

    The fill tape is read from ``raw_tape_path`` (raw on-chain individual fills).
    Orders are simulated chronologically against the tape so that each tape print's
    quantity can only be consumed once across all active copy orders in the shard.
    """
    TRIG_COLS = [
        "wallet", "condition_id", "outcome", "dt", "side",
        "avg_price", "total_quantity", "trade_pnl", "token_winner",
        "tx_hash",
    ]
    tdf = pd.read_parquet(shard_path, columns=TRIG_COLS)
    if tdf.empty:
        return pd.DataFrame()

    tdf["dt"] = pd.to_datetime(tdf["dt"], utc=True)
    tdf = tdf.rename(columns={"avg_price": "price", "total_quantity": "quantity"})

    date_mask = (
        (tdf["dt"].dt.date >= period_start)
        & (tdf["dt"].dt.date <= period_end)
    )
    tdf = tdf[date_mask].copy()
    if tdf.empty:
        return pd.DataFrame()

    tdf["price"] = tdf["price"].astype(float)
    tdf["quantity"] = tdf["quantity"].astype(float)

    if ONLY_BUY_TRIGGERS:
        trigger_mask = (tdf["wallet"].isin(watched_wallets)) & (tdf["side"] == "BUY")
    else:
        trigger_mask = tdf["wallet"].isin(watched_wallets)
    triggers_df = tdf[trigger_mask].copy()
    if triggers_df.empty:
        return pd.DataFrame()

    tw_map: dict[tuple[str, str], bool] = {}
    for row in tdf[["condition_id", "outcome", "token_winner"]].itertuples(index=False):
        key = (row.condition_id, row.outcome)
        if key not in tw_map and row.token_winner is not None:
            tw_map[key] = bool(row.token_winner)

    complement = _build_complement_map(tdf)
    triggers_df = triggers_df.sort_values("dt", kind="mergesort").reset_index(drop=True)

    TAPE_COLS = ["condition_id", "outcome", "block_timestamp", "side", "price", "quantity", "tx_hash", "token_winner"]
    if raw_tape_path.exists():
        rdf = pd.read_parquet(raw_tape_path, columns=TAPE_COLS)
    else:
        rdf = pd.DataFrame(columns=TAPE_COLS)

    if rdf.empty:
        ledger_rows = []
        for trig in triggers_df.itertuples(index=False):
            cid = trig.condition_id
            outcome = trig.outcome
            side = trig.side
            trig_dt = trig.dt
            trig_px = float(trig.price)
            trig_qty = float(trig.quantity)
            trig_tw = bool(trig.token_winner)
            wallet = trig.wallet
            if side == "BUY":
                order_px = float(np.clip(trig_px + worse_price_offset, 0.001, 0.999))
                qty_ordered = min(trig_qty, stake_usdc / order_px)
            else:
                order_px = float(np.clip(trig_px - worse_price_offset, 0.001, 0.999))
                qty_ordered = trig_qty
            ledger_rows.append({
                "row_type": "trigger", "order_id": len(ledger_rows) + 1,
                "wallet": wallet, "condition_id": cid, "outcome": outcome,
                "side": side, "dt": trig_dt, "tx_hash": trig.tx_hash,
                "price": trig_px, "order_price": order_px,
                "qty_ordered": qty_ordered, "qty_filled": 0.0,
                "fill_qty": None, "fill_source": None,
                "token_winner": trig_tw, "exec_price": None,
                "fill_pnl_usdc": None,
                "wallet_trade_pnl": float(trig.trade_pnl) if trig.trade_pnl is not None else None,
                "filled_position": 0.0,
            })
        result = pd.DataFrame(ledger_rows)
        result["shard"] = shard_path.stem
        return result

    rdf["dt"] = pd.to_datetime(rdf["block_timestamp"], unit="s", utc=True)
    rdf = rdf.drop(columns=["block_timestamp"])

    tape_start = pd.Timestamp(period_start, tz="UTC")
    tape_end = pd.Timestamp(period_end, tz="UTC") + pd.Timedelta(days=1)
    rdf = rdf[(rdf["dt"] >= tape_start) & (rdf["dt"] < tape_end)].copy()
    rdf["price"] = rdf["price"].astype(float)
    rdf["quantity"] = rdf["quantity"].astype(float)
    rdf = rdf.sort_values("dt", kind="mergesort").reset_index(drop=True)

    fee = fee_bps / 10_000.0
    horizon = pd.Timedelta(seconds=fill_horizon_seconds)
    eps = 1e-12

    ledger_rows: list[dict] = []
    order_counter = 0
    # (wallet, condition_id, outcome) → filled qty for this outcome
    positions: dict[tuple[str, str, str], float] = defaultdict(float)
    # (wallet, condition_id) → total filled qty across all outcomes of that condition
    cond_positions: dict[tuple[str, str], float] = defaultdict(float)
    # wallet → total filled qty across all conditions
    wallet_positions: dict[str, float] = defaultdict(float)

    orders: dict[int, dict] = {}
    books: dict[tuple[str, str, str], list[dict]] = defaultdict(list)

    def _append_trigger_row(order_id: int, trig, order_px: float, qty_ordered: float, trig_tw: bool, current_pos: float = 0.0) -> None:
        ledger_rows.append({
            "row_type": "trigger",
            "order_id": order_id,
            "wallet": trig.wallet,
            "condition_id": trig.condition_id,
            "outcome": trig.outcome,
            "side": trig.side,
            "dt": trig.dt,
            "tx_hash": trig.tx_hash,
            "price": float(trig.price),
            "order_price": order_px,
            "qty_ordered": qty_ordered,
            "qty_filled": None,
            "fill_qty": None,
            "fill_source": None,
            "token_winner": trig_tw,
            "exec_price": None,
            "fill_pnl_usdc": None,
            "wallet_trade_pnl": float(trig.trade_pnl) if trig.trade_pnl is not None else None,
            "filled_position": current_pos,
        })

    # order can match with trade on the same side and token, or opposite side and opposite token    
    def _register_order(order_id: int, order: dict, trig_tw: bool) -> None:
        cid = order["condition_id"]
        outcome = order["outcome"]
        side = order["side"]
        books[(cid, outcome, side)].append({
            "order_id": order_id,
            "fill_source": "same_token",
            "fill_tw": trig_tw,
            "opposite": False,
        })
        opp_outcome = complement.get((cid, outcome))
        if opp_outcome is None:
            return
        opp_tw = tw_map.get((cid, opp_outcome), not trig_tw)
        opp_tape_side = "SELL" if side == "BUY" else "BUY"
        books[(cid, opp_outcome, opp_tape_side)].append({
            "order_id": order_id,
            "fill_source": "opposite_token",
            "fill_tw": trig_tw,
            "opposite": True,
        })

    def _process_tape_row(row) -> None:
        key = (row.condition_id, row.outcome, row.side)
        entries = books.get(key)
        if not entries:
            return

        tape_dt = row.dt
        tape_price = float(row.price)
        tape_remaining = float(row.quantity)
        if tape_remaining <= eps:
            return

        survivors: list[dict] = []
        for entry in entries:
            order = orders.get(entry["order_id"])
            if order is None:
                continue
            if order["remaining_qty"] <= eps:
                continue
            if order["deadline"] < tape_dt:
                continue

            eff_price = float(np.clip(1.0 - tape_price, 0.001, 0.999)) if entry["opposite"] else tape_price
            eligible = eff_price <= order["order_price"] if order["side"] == "BUY" else eff_price >= order["order_price"]

            if eligible and tape_remaining > eps:
                fill_qty = min(order["remaining_qty"], tape_remaining)

                # Cap fill to remaining position headroom (per-condition and per-wallet limits).
                # This prevents overfill when a single tape print is large enough to exceed the cap.
                if order["side"] == "BUY":
                    if max_position_per_condition is not None:
                        cond_headroom = max(max_position_per_condition - cond_positions[(order["wallet"], order["condition_id"])], 0.0)
                        fill_qty = min(fill_qty, cond_headroom)
                    if max_position_per_wallet is not None:
                        wallet_headroom = max(max_position_per_wallet - wallet_positions[order["wallet"]], 0.0)
                        fill_qty = min(fill_qty, wallet_headroom)
                    if fill_qty <= eps:
                        # Position cap already reached — retire this order without consuming tape
                        order["remaining_qty"] = 0.0
                        continue

                tape_remaining -= fill_qty
                order["remaining_qty"] -= fill_qty

                if order["side"] == "BUY":
                    gross = fill_qty * (1.0 - order["order_price"]) if entry["fill_tw"] else -fill_qty * order["order_price"]
                    fill_pnl = gross - fill_qty * order["order_price"] * fee
                    positions[order["pos_key"]] += fill_qty
                    cond_positions[(order["wallet"], order["condition_id"])] += fill_qty
                    wallet_positions[order["wallet"]] += fill_qty
                    new_pos = positions[order["pos_key"]]
                else:
                    gross = fill_qty * (order["order_price"] - 1.0) if entry["fill_tw"] else fill_qty * order["order_price"]
                    fill_pnl = gross - fill_qty * order["order_price"] * fee
                    prev = positions[order["pos_key"]]
                    positions[order["pos_key"]] = max(prev - fill_qty, 0.0)
                    actually_reduced = prev - positions[order["pos_key"]]
                    cond_positions[(order["wallet"], order["condition_id"])] = max(
                        cond_positions[(order["wallet"], order["condition_id"])] - actually_reduced, 0.0
                    )
                    wallet_positions[order["wallet"]] = max(
                        wallet_positions[order["wallet"]] - actually_reduced, 0.0
                    )
                    new_pos = positions[order["pos_key"]]

                ledger_rows.append({
                    "row_type": "fill",
                    "order_id": entry["order_id"],
                    "wallet": order["wallet"],
                    "condition_id": order["condition_id"],
                    "outcome": order["outcome"],
                    "side": order["side"],
                    "dt": tape_dt,
                    "tx_hash": row.tx_hash,
                    "price": eff_price,
                    "order_price": None,
                    "qty_ordered": order["qty_ordered"],
                    "qty_filled": None,
                    "fill_qty": fill_qty,
                    "fill_source": entry["fill_source"],
                    "token_winner": entry["fill_tw"],
                    "exec_price": order["order_price"],
                    "fill_pnl_usdc": fill_pnl,
                    "wallet_trade_pnl": None,
                    "filled_position": new_pos,
                })

            if order["remaining_qty"] > eps and order["deadline"] >= tape_dt:
                survivors.append(entry)

        if survivors:
            books[key] = survivors
        else:
            books.pop(key, None)

    tape_iter = iter(rdf[["condition_id", "outcome", "dt", "side", "price", "quantity", "tx_hash", "token_winner"]].itertuples(index=False))
    current_tape = next(tape_iter, None)

    for trig in triggers_df.itertuples(index=False):
        while current_tape is not None and current_tape.dt <= trig.dt:
            _process_tape_row(current_tape)
            current_tape = next(tape_iter, None)

        cid = trig.condition_id
        outcome = trig.outcome
        side = trig.side
        trig_px = float(trig.price)
        trig_qty = float(trig.quantity)
        trig_tw = bool(trig.token_winner)
        wallet = trig.wallet
        pos_key = (wallet, cid, outcome)

        if side == "BUY":
            order_px = float(np.clip(trig_px + worse_price_offset, 0.001, 0.999))
            current_pos = positions.get(pos_key, 0.0)
            qty_ordered = min(trig_qty, stake_usdc / order_px)
        else:
            order_px = float(np.clip(trig_px - worse_price_offset, 0.001, 0.999))
            current_pos = positions.get(pos_key, 0.0)
            if current_pos <= eps:
                continue
            sell_cap = stake_usdc / max(1.0 - order_px, 1e-9)
            qty_ordered = min(trig_qty, current_pos, sell_cap)
            if qty_ordered <= eps:
                continue

        order_counter += 1
        order_id = order_counter

        order = {
            "wallet": wallet,
            "condition_id": cid,
            "outcome": outcome,
            "side": side,
            "order_price": order_px,
            "qty_ordered": qty_ordered,
            "remaining_qty": qty_ordered,
            "deadline": trig.dt + horizon,
            "pos_key": pos_key,
        }
        orders[order_id] = order

        _append_trigger_row(order_id, trig, order_px, qty_ordered, trig_tw, current_pos=positions.get(pos_key, 0.0))
        _register_order(order_id, order, trig_tw)

    while current_tape is not None:
        _process_tape_row(current_tape)
        current_tape = next(tape_iter, None)

    if not ledger_rows:
        return pd.DataFrame()

    result = pd.DataFrame(ledger_rows)
    result["shard"] = shard_path.stem

    filled_by_order = (
        result[result["row_type"] == "fill"]
        .groupby("order_id")["fill_qty"]
        .sum()
        .rename("_total_filled")
    )
    result = result.merge(filled_by_order, on="order_id", how="left")
    trig_mask = result["row_type"] == "trigger"
    result.loc[trig_mask, "qty_filled"] = result.loc[trig_mask, "_total_filled"].fillna(0.0)
    result = result.drop(columns=["_total_filled"])
    result["qty_filled"] = result["qty_filled"].astype(float)

    return result



## Run backtest across all shards (parallel)

In [6]:
assert WATCHED_WALLETS, "WATCHED_WALLETS is empty — set wallets in the config cell or run the wallet-load cell."

watched_set = set(WATCHED_WALLETS)
shard_results: list[pd.DataFrame] = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {
        executor.submit(
            _simulate_shard,
            path,
            RAW_TRADES_DIR / path.name,
            watched_set,
            period_start,
            period_end,
            WORSE_PRICE_OFFSET,
            FILL_HORIZON_SECONDS,
            STAKE_USDC,
            FEE_BPS,
            MAX_POSITION_PER_CONDITION,
            MAX_POSITION_PER_WALLET,
        ): path
        for path in SHARD_PATHS
    }
    for future in as_completed(futures):
        path = futures[future]
        try:
            df = future.result()
            if df is not None and not df.empty:
                shard_results.append(df)
        except Exception as exc:
            print(f"Shard {path.name} raised an exception: {exc}")
            raise

if shard_results:
    event_ledger: pd.DataFrame = (
        pd.concat(shard_results, ignore_index=True)
        .sort_values(["dt", "shard", "order_id", "row_type"])
        .reset_index(drop=True)
    )
    _key = event_ledger[["shard", "order_id"]]
    _pairs = _key.drop_duplicates().reset_index(drop=True)
    _pairs["global_order_id"] = _pairs.index + 1
    event_ledger = event_ledger.merge(_pairs, on=["shard", "order_id"], how="left")
    event_ledger["order_id"] = event_ledger["global_order_id"]
    event_ledger = event_ledger.drop(columns=["global_order_id"])
else:
    event_ledger = pd.DataFrame(columns=[
        "row_type", "order_id", "wallet", "condition_id", "outcome",
        "side", "dt", "tx_hash", "price", "order_price",
        "qty_ordered", "qty_filled", "fill_qty",
        "fill_source", "token_winner", "exec_price", "fill_pnl_usdc",
        "wallet_trade_pnl", "shard", "filled_position",
    ])

triggers = event_ledger[event_ledger["row_type"] == "trigger"]
fills = event_ledger[event_ledger["row_type"] == "fill"]
filled_orders = fills["order_id"].nunique()

print(f"Trigger events    : {len(triggers):>7,}")
print(f"Fill rows         : {len(fills):>7,}")
print(f"Orders with fills : {filled_orders:>7,}")
if len(triggers) > 0:
    print(f"Order fill rate   : {filled_orders/len(triggers)*100:.1f}%")
else:
    print("No trigger events found — check WATCHED_WALLETS and the period.")



Trigger events    : 187,382
Fill rows         :  13,842
Orders with fills :  10,701
Order fill rate   : 5.7%


## Summary statistics

In [7]:
fee = FEE_BPS / 10_000.0

if not fills.empty:
    filled_copy_pnl    = fills["fill_pnl_usdc"].sum()
    total_qty_filled   = fills["fill_qty"].sum()
    total_notional     = (fills["fill_qty"] * fills["exec_price"]).sum()
    orders_with_fills  = fills["order_id"].nunique()
    partial_orders     = (fills.groupby("order_id").size() > 1).sum()
    win_rate           = fills.groupby("order_id")["token_winner"].first().mean()
    avg_exec_price     = fills["exec_price"].mean()
    fill_src_counts    = fills["fill_source"].value_counts()

    # Partial-fill statistics
    trig_qty = triggers.set_index("order_id")["qty_ordered"]
    fill_qty = triggers.set_index("order_id")["qty_filled"]
    fill_pct = (fill_qty / trig_qty.clip(lower=1e-12) * 100).replace([float('inf'), float('nan')], 0)

    print(f"=== Copy-strategy performance ===")
    print(f"  Orders triggered    : {len(triggers):>7,}")
    print(f"  Orders with fills   : {orders_with_fills:>7,}  ({orders_with_fills/len(triggers)*100:.1f}%)")
    print(f"  Orders multi-fill   : {partial_orders:>7,}  ({partial_orders/max(orders_with_fills,1)*100:.1f}% of filled)")
    print(f"  Avg fill %          : {fill_pct[fill_pct>0].mean():>7.1f}%")
    print(f"  Total qty filled    : {total_qty_filled:>10,.2f} tokens")
    print(f"  Total notional      : {total_notional:>10,.2f} USDC")
    print(f"  Net PnL (USDC)      : {filled_copy_pnl:>10,.2f}")
    print(f"  Net ROI on notional : {filled_copy_pnl/total_notional*100:>8.2f}%")
    print(f"  Win rate (by order) : {win_rate*100:>8.2f}%")
    print(f"  Avg exec price      : {avg_exec_price:>8.4f}")
    print(f"\n  Fill source breakdown:")
    for src, cnt in fill_src_counts.items():
        print(f"    {src:<20s}: {cnt:>6,}  ({cnt/len(fills)*100:.1f}%)")
else:
    print("No fills — nothing to summarise.")

# Watched-wallet cohort summary
wallet_pnl = triggers["wallet_trade_pnl"].sum()
print(f"\n=== Watched-wallet cohort ===")
print(f"  Total trades        : {len(triggers):>7,}")
print(f"  Total trade PnL     : {wallet_pnl:>10,.2f} USDC")


=== Copy-strategy performance ===
  Orders triggered    : 187,382
  Orders with fills   :  10,701  (5.7%)
  Orders multi-fill   :   1,964  (18.4% of filled)
  Avg fill %          :    74.9%
  Total qty filled    :  66,786.65 tokens
  Total notional      :  40,092.04 USDC
  Net PnL (USDC)      :     730.96
  Net ROI on notional :     1.82%
  Win rate (by order) :    60.67%
  Avg exec price      :   0.5882

  Fill source breakdown:
    same_token          : 10,461  (75.6%)
    opposite_token      :  3,381  (24.4%)

=== Watched-wallet cohort ===
  Total trades        : 187,382
  Total trade PnL     : 3,336,169.87 USDC


## Event ledger preview

In [8]:
event_ledger[
    (event_ledger['row_type'] == 'fill')
    & (event_ledger['condition_id'] == '0x3488f31e6449f9803f99a8b5dd232c7ad883637f1c86e6953305a2ef19c77f20')
    & (event_ledger['wallet'] == '0x0b219cf3d297991b58361dbebdbaa91e56b8deb6')
    ].head(500)

,row_type,order_id,wallet,condition_id,outcome,side,dt,tx_hash,price,order_price,qty_ordered,qty_filled,fill_qty,fill_source,token_winner,exec_price,fill_pnl_usdc,wallet_trade_pnl,filled_position,shard
392,fill,297,0x0b219cf3d297991b58361dbebdbaa91e56b8deb6,0x3488f31e6449f9803f99a8b5dd232c7ad883637f1c86...,No,BUY,2026-02-16 03:52:45+00:00,0x35be21e50c08c48f6f6fbb87ff3ab0cc799b7b9609e5...,0.86,NaN,11.627907,NaN,7.750000,same_token,False,0.86,-6.671665,NaN,7.750000,3
394,fill,297,0x0b219cf3d297991b58361dbebdbaa91e56b8deb6,0x3488f31e6449f9803f99a8b5dd232c7ad883637f1c86...,No,BUY,2026-02-16 03:52:59+00:00,0xad8b81708c6197f05eaa0efb8866175743b49675dd24...,0.86,NaN,11.627907,NaN,2.250000,same_token,False,0.86,-1.936935,NaN,10.000000,3
68814,fill,61262,0x0b219cf3d297991b58361dbebdbaa91e56b8deb6,0x3488f31e6449f9803f99a8b5dd232c7ad883637f1c86...,No,SELL,2026-02-28 06:23:21+00:00,0xde3861873a15e671c4c9fe862015a5d9e041b73a7610...,0.33,NaN,10.000000,NaN,10.000000,same_token,False,0.33,3.296700,NaN,0.000000,3
69579,fill,61958,0x0b219cf3d297991b58361dbebdbaa91e56b8deb6,0x3488f31e6449f9803f99a8b5dd232c7ad883637f1c86...,Yes,BUY,2026-02-28 06:46:45+00:00,0x242b52d0eb424c9ff18b6193e0c0f45a630950efba70...,0.87,NaN,11.363636,NaN,10.000000,opposite_token,True,0.88,1.191200,NaN,10.000000,3
69777,fill,62138,0x0b219cf3d297991b58361dbebdbaa91e56b8deb6,0x3488f31e6449f9803f99a8b5dd232c7ad883637f1c86...,Yes,SELL,2026-02-28 06:50:31+00:00,0x8e3717d3d9061e2e7cae2fb50a7b7e18414779073334...,0.92,NaN,10.000000,NaN,0.680000,same_token,True,0.92,-0.055026,NaN,9.320000,3
69778,fill,62138,0x0b219cf3d297991b58361dbebdbaa91e56b8deb6,0x3488f31e6449f9803f99a8b5dd232c7ad883637f1c86...,Yes,SELL,2026-02-28 06:50:31+00:00,0x08d09aee3c7c7835b4990dcc36757cc4ada142b34113...,0.92,NaN,10.000000,NaN,0.183637,same_token,True,0.92,-0.014860,NaN,9.136363,3
69779,fill,62138,0x0b219cf3d297991b58361dbebdbaa91e56b8deb6,0x3488f31e6449f9803f99a8b5dd232c7ad883637f1c86...,Yes,SELL,2026-02-28 06:50:31+00:00,0x08d09aee3c7c7835b4990dcc36757cc4ada142b34113...,0.92,NaN,10.000000,NaN,1.366363,opposite_token,True,0.92,-0.110566,NaN,7.770000,3
69780,fill,62138,0x0b219cf3d297991b58361dbebdbaa91e56b8deb6,0x3488f31e6449f9803f99a8b5dd232c7ad883637f1c86...,Yes,SELL,2026-02-28 06:50:31+00:00,0x731390f766620e01519409767259574028e2bb94f5f9...,0.92,NaN,10.000000,NaN,7.770000,same_token,True,0.92,-0.628748,NaN,0.000000,3
69781,fill,62139,0x0b219cf3d297991b58361dbebdbaa91e56b8deb6,0x3488f31e6449f9803f99a8b5dd232c7ad883637f1c86...,No,BUY,2026-02-28 06:50:31+00:00,0x731390f766620e01519409767259574028e2bb94f5f9...,0.08,NaN,0.006449,NaN,0.006449,opposite_token,False,0.08,-0.000516,NaN,0.006449,3
69804,fill,62156,0x0b219cf3d297991b58361dbebdbaa91e56b8deb6,0x3488f31e6449f9803f99a8b5dd232c7ad883637f1c86...,Yes,BUY,2026-02-28 06:51:19+00:00,0x09fa8bcbe8fe2b7ceb46b9e2b0439a790964f7eb33d5...,0.91,NaN,10.989010,NaN,9.993551,opposite_token,True,0.91,0.890325,NaN,9.993551,3


In [9]:
pd.set_option('display.max_rows', 100)

In [10]:
display_cols = [
    "row_type", "order_id", "wallet", "condition_id", "outcome", "side",
    "dt", "tx_hash", "price", "order_price", "exec_price",
    "qty_ordered", "qty_filled", "fill_qty",
    "fill_source", "token_winner", "fill_pnl_usdc", "filled_position", "wallet_trade_pnl",
]
available = [c for c in display_cols if c in event_ledger.columns]

# Show a few trigger+fill pairs
sample_orders = event_ledger["order_id"].unique()[:5]
event_ledger[
    event_ledger["order_id"].isin(sample_orders)
][available].sort_values(["order_id", "row_type"])


,row_type,order_id,wallet,condition_id,outcome,side,dt,tx_hash,price,order_price,exec_price,qty_ordered,qty_filled,fill_qty,fill_source,token_winner,fill_pnl_usdc,filled_position,wallet_trade_pnl
2,fill,1,0xd6eef76188c7068719ac881460b7431e1ae137a2,0xf86191c4b660e0f0f4669eae5638474277a3bb31e5c3...,No,BUY,2026-02-16 00:00:51+00:00,0x0b8f56c045cc516e71896d5cf7f99a83674e137e5192...,0.960,NaN,0.961,0.004241,NaN,0.004241,same_token,True,0.000161,0.004241,NaN
0,trigger,1,0xd6eef76188c7068719ac881460b7431e1ae137a2,0xf86191c4b660e0f0f4669eae5638474277a3bb31e5c3...,No,BUY,2026-02-16 00:00:15+00:00,0x0b3d6d10434abd444fe1510c39f55ce99398e88c8962...,0.961,0.961,NaN,0.004241,0.004241,NaN,None,True,NaN,0.000000,0.000165
1,trigger,2,0xc7b8a3f1033a9efa9d912b60885c1d2c1e012c1f,0xbf3a613ba4ab4a066c2c3eacb877175d39a93f1fde88...,Down,BUY,2026-02-16 00:00:33+00:00,0x0ba17729dac95a428ae9e98c9c1a4004006ca120f47e...,0.410,0.410,NaN,24.390244,0.000000,NaN,None,False,NaN,0.000000,-492.000000
4,fill,3,0xd6eef76188c7068719ac881460b7431e1ae137a2,0xbebb1c8c6996bdd0783db8891f4cb764afe301c7393a...,Yes,BUY,2026-02-16 00:01:41+00:00,0xf2f6a76c3c0c2a1d79d01aaf557e648c61ec3c988d28...,0.910,NaN,0.910,2.170000,NaN,2.170000,same_token,True,0.193325,2.170000,NaN
3,trigger,3,0xd6eef76188c7068719ac881460b7431e1ae137a2,0xbebb1c8c6996bdd0783db8891f4cb764afe301c7393a...,Yes,BUY,2026-02-16 00:00:55+00:00,0xc0a6b59be75f38f289ce014c517b4a661ba472d8ed97...,0.910,0.910,NaN,2.170000,2.170000,NaN,None,True,NaN,0.000000,0.195300
6,fill,4,0xd6eef76188c7068719ac881460b7431e1ae137a2,0xbebb1c8c6996bdd0783db8891f4cb764afe301c7393a...,Yes,BUY,2026-02-16 00:02:09+00:00,0xca8ef7824274e6e5e97d02f83f95d06d1985eacac3ba...,0.910,NaN,0.910,2.170000,NaN,2.140000,same_token,True,0.190653,4.310000,NaN
5,trigger,4,0xd6eef76188c7068719ac881460b7431e1ae137a2,0xbebb1c8c6996bdd0783db8891f4cb764afe301c7393a...,Yes,BUY,2026-02-16 00:01:41+00:00,0xf2f6a76c3c0c2a1d79d01aaf557e648c61ec3c988d28...,0.910,0.910,NaN,2.170000,2.140000,NaN,None,True,NaN,2.170000,0.195300
7,trigger,5,0xd6eef76188c7068719ac881460b7431e1ae137a2,0xbebb1c8c6996bdd0783db8891f4cb764afe301c7393a...,Yes,BUY,2026-02-16 00:02:09+00:00,0xca8ef7824274e6e5e97d02f83f95d06d1985eacac3ba...,0.910,0.910,NaN,2.140000,0.000000,NaN,None,True,NaN,4.310000,0.192600


In [11]:
event_ledger[event_ledger['side'] == 'BUY'].groupby('wallet').agg(
    fill_pnl_usdc_sum=('fill_pnl_usdc', 'sum'),
    wallet_trade_pnl_sum=('wallet_trade_pnl', 'sum'),
    buy_count=('side', 'count')
)

,fill_pnl_usdc_sum,wallet_trade_pnl_sum,buy_count
wallet,,,
0x000d257d2dc7616feaef4ae0f14600fdf50a758e,17.215606,50109.836807,809
0x023c300ddb1b8931448ff4371be11fce9467805d,-14.154080,-1296.231633,41
0x027a422d069cde53d7ff6baf4ed3212d7a23ea13,10.486166,977.556457,101
0x06ecb7e739f5455922ce57e83284f132c7f0f845,-7.307300,-7300.000000,2
0x08c897e4780c299061fe683120e5bb18cf931da2,11.971876,1231.812014,114
...,...,...,...
0xf38e6583dfa37026ee25cf0a567b8cef6307a38a,3.474606,17255.472050,124
0xf41b55cf7d9fbe2fcf4895ab5f71d8644ef2b538,-6.296230,-3729.590361,114
0xf658449199d0bcf544de0ff928a3b66685f3dcfe,28.732471,2818.153878,211


In [12]:
event_ledger[event_ledger['side'] == 'BUY'].groupby('condition_id').agg(
    fill_pnl_usdc_sum=('fill_pnl_usdc', 'sum'),
    wallet_trade_pnl_sum=('wallet_trade_pnl', 'sum'),
    buy_count=('side', 'count')
).sort_values('fill_pnl_usdc_sum', ascending=False).head(10)

,fill_pnl_usdc_sum,wallet_trade_pnl_sum,buy_count
condition_id,,,
0x3488f31e6449f9803f99a8b5dd232c7ad883637f1c86e6953305a2ef19c77f20,246.025609,1.168118e+06,4128
0x4b02efe53e631ada84681303fd66d79ad615f3d2b6a28b4633d43d935f89af58,108.350888,-2.251410e+03,661
0x61ce3773237a948584e422de72265f937034af418a8b703e3a860ea62e59ff36,90.142132,2.361633e+05,3467
0x4b7cd1050a0c87baebe5e6008260d14b2a9d6d705cf7e7daac1be94ad5253b27,75.024492,5.712536e+04,1465
0x86d302f6dd0e23ae1ee8ef7d36c8fd3a1f0f7c08fbeb4ec0f2c6e0b85b9cc2cf,62.315035,1.324891e+04,493
0x77508b9cb5962ff587890f79e3e39a1789384405b6ff530afbccb9d70743352a,48.896513,-2.872452e+04,660
0x09cbe3e796661a1d820580145488ad2ccb9ad1e720efcd64b448bb77b97007c1,48.609326,2.538654e+04,1573
0x43edd40756d4dfaa633687ddbfab445cdd21f9083a45289871e95b5292793bca,44.424285,9.606163e+03,1667
0x28f1e9218bcd41ced05faf3c03f026721d87712556bf2828337551f03cae0a01,41.365787,2.732299e+04,261


In [13]:
event_ledger[event_ledger['side'] == 'SELL'].groupby('wallet').agg(
    fill_pnl_usdc_sum=('fill_pnl_usdc', 'sum'),
    wallet_trade_pnl_sum=('wallet_trade_pnl', 'sum'),
    sell_count=('side', 'count')
)

,fill_pnl_usdc_sum,wallet_trade_pnl_sum,sell_count
wallet,,,
0x000d257d2dc7616feaef4ae0f14600fdf50a758e,-0.842221,-1316.185720,24
0x023c300ddb1b8931448ff4371be11fce9467805d,16.483500,1174.140000,4
0x027a422d069cde53d7ff6baf4ed3212d7a23ea13,-12.205422,-465.649655,19
0x0a793254e040027c5d1ca0c2f87cda8e22228e05,2.517427,720.805356,22
0x0b219cf3d297991b58361dbebdbaa91e56b8deb6,30.310189,858.971379,38
...,...,...,...
0xeede126b67fc64bc36ca5adea94531ebb3708a73,-0.243712,-63.022760,6
0xf41b55cf7d9fbe2fcf4895ab5f71d8644ef2b538,-0.509530,-57.315252,8
0xf658449199d0bcf544de0ff928a3b66685f3dcfe,2.015568,87.328400,7


In [52]:
(fills[fills['condition_id'] == '0x3488f31e6449f9803f99a8b5dd232c7ad883637f1c86e6953305a2ef19c77f20'][['dt', 'side', 'outcome', 'price', 'fill_qty', 'exec_price', 'filled_position', 'order_id', 'tx_hash']]
 .sort_values('dt')
 .head(100))

,dt,side,outcome,price,fill_qty,exec_price,filled_position,order_id,tx_hash
60,2026-02-16 00:54:17+00:00,BUY,No,0.86,10.000000,0.860000,10.000000,38,0x65e3bc1e9703026f77126a71f231e5468e505c11dbffe14868849e735025806b
145,2026-02-16 01:41:35+00:00,BUY,Yes,0.14,10.000000,0.140000,10.000000,116,0x08d6306ab5d6b46d6b1e54aad9a98f83d584966cb7fc0de91f2a884c6ca3b76f
392,2026-02-16 03:52:45+00:00,BUY,No,0.86,7.750000,0.860000,7.750000,297,0x35be21e50c08c48f6f6fbb87ff3ab0cc799b7b9609e596f0e9e556f6d9e3af2f
394,2026-02-16 03:52:59+00:00,BUY,No,0.86,2.250000,0.860000,10.000000,297,0xad8b81708c6197f05eaa0efb8866175743b49675dd24640221280c3728ef88ec
830,2026-02-16 05:35:45+00:00,SELL,Yes,0.15,10.000000,0.147926,0.000000,608,0x5fd58375b12b069d53746b78cda1e9ed354081f59928d80b503f0a9fbba57b10
1088,2026-02-16 07:50:21+00:00,BUY,Yes,0.14,10.000000,0.140000,10.000000,817,0xd3830552a16edd730db0e09b4e0be0774e01027a9b1761aa1c4a6f21b0bb162d
1140,2026-02-16 08:06:47+00:00,SELL,Yes,0.16,10.000000,0.160000,0.000000,859,0x877d1aa90e6275db23e755b1b00104cf281999585633d4bf2aea7650d0901357
1285,2026-02-16 09:38:23+00:00,BUY,No,0.86,10.000000,0.860000,10.000000,986,0x4d3c68159e6933799d9e2ebc3f448efb7c3da5d2b1f1abd78f3a8a20ea459d5d
1345,2026-02-16 09:55:31+00:00,BUY,No,0.86,10.000000,0.860000,10.000000,1027,0x32a45534057820450d706519b7d439d1e8482074a65b671c5bce9da9767206e5
1571,2026-02-16 12:05:23+00:00,BUY,No,0.87,10.000000,0.870000,10.000000,1216,0xa12cdcea586eb71103de733afab69e5aa26c784c618eca3ff704b82020f1e478


In [14]:
sample_filled_orders = fills["order_id"].unique()[:50]
event_ledger[
    (event_ledger['side'] == 'SELL') &
    (event_ledger["order_id"].isin(sample_filled_orders))
].sort_values(["order_id", "dt"])

,row_type,order_id,wallet,condition_id,outcome,side,dt,tx_hash,price,order_price,qty_ordered,qty_filled,fill_qty,fill_source,token_winner,exec_price,fill_pnl_usdc,wallet_trade_pnl,filled_position,shard
137,trigger,111,0xd6eef76188c7068719ac881460b7431e1ae137a2,0xbebb1c8c6996bdd0783db8891f4cb764afe301c7393a...,Yes,SELL,2026-02-16 01:36:57+00:00,0xe799872a64d88d4f5007f0cef239a8e2025062a9a4c3...,0.930,0.930,4.310000,3.220000,NaN,None,True,NaN,NaN,-0.474277,4.310000,b
143,fill,111,0xd6eef76188c7068719ac881460b7431e1ae137a2,0xbebb1c8c6996bdd0783db8891f4cb764afe301c7393a...,Yes,SELL,2026-02-16 01:41:01+00:00,0x28205668d943eae092b5941958aff44e9444eae59fc3...,0.930,NaN,4.310000,NaN,3.220000,same_token,True,0.930,-0.228395,NaN,1.090000,b
170,trigger,139,0xd6eef76188c7068719ac881460b7431e1ae137a2,0x7a310d9af29782a53ccb16cead33ff3140a696c2bc40...,No,SELL,2026-02-16 02:06:13+00:00,0x8142dd1fcfed2b6297e95b340ae0753d64d6c5b5c1b6...,0.780,0.780,0.027875,0.027875,NaN,None,True,NaN,NaN,-0.006133,0.027875,7
172,fill,139,0xd6eef76188c7068719ac881460b7431e1ae137a2,0x7a310d9af29782a53ccb16cead33ff3140a696c2bc40...,No,SELL,2026-02-16 02:07:49+00:00,0xda84d04784a87bd05430e8d86fd1b10479db07c4ab60...,0.790,NaN,0.027875,NaN,0.027875,opposite_token,True,0.780,-0.006154,NaN,0.000000,7
219,trigger,179,0xd6eef76188c7068719ac881460b7431e1ae137a2,0x382aed04fd32aa35d057e5a08e270f0176d2e5b2a178...,No,SELL,2026-02-16 02:52:13+00:00,0x20f016e82a014d912845ce9fb1d1d2d8c074e5504f67...,0.780,0.780,2.565941,2.565941,NaN,None,True,NaN,NaN,-0.564507,2.565941,3
222,fill,179,0xd6eef76188c7068719ac881460b7431e1ae137a2,0x382aed04fd32aa35d057e5a08e270f0176d2e5b2a178...,No,SELL,2026-02-16 02:52:33+00:00,0x7690ca3f73a0c95047a3340f685eb6548566b2766cc1...,0.790,NaN,2.565941,NaN,2.565941,opposite_token,True,0.780,-0.566508,NaN,0.000000,3
232,trigger,186,0xd6eef76188c7068719ac881460b7431e1ae137a2,0x1597a8e275ae13b3be4a1875362a44056bdd5a0b72ca...,Yes,SELL,2026-02-16 02:56:27+00:00,0x2a7aa900971ce691859988648f4d3df5bf3296b61b5a...,0.930,0.930,8.010751,8.010751,NaN,None,True,NaN,NaN,-0.560753,10.000000,1
236,fill,186,0xd6eef76188c7068719ac881460b7431e1ae137a2,0x1597a8e275ae13b3be4a1875362a44056bdd5a0b72ca...,Yes,SELL,2026-02-16 02:58:27+00:00,0x71621b052389dd7a9f7855c663573b4e286bb13b693f...,0.930,NaN,8.010751,NaN,8.010751,same_token,True,0.930,-0.568203,NaN,1.989249,1
237,trigger,189,0xd6eef76188c7068719ac881460b7431e1ae137a2,0x1597a8e275ae13b3be4a1875362a44056bdd5a0b72ca...,Yes,SELL,2026-02-16 02:58:27+00:00,0xe4a3a60d9d6c25246a09e2ac735ac32f2dd3667d55d4...,0.912,0.912,1.989249,1.989249,NaN,None,True,NaN,NaN,-2.640000,1.989249,1
238,fill,189,0xd6eef76188c7068719ac881460b7431e1ae137a2,0x1597a8e275ae13b3be4a1875362a44056bdd5a0b72ca...,Yes,SELL,2026-02-16 02:58:29+00:00,0x14e77060d9a4b19b4f37874cdb7f21405994ffac85ba...,0.930,NaN,1.989249,NaN,1.989249,opposite_token,True,0.912,-0.176868,NaN,0.000000,1


## Build PnL time series

In [15]:
def resample_hourly_series(df: pd.DataFrame, dt_col: str, pnl_col: str) -> pd.DataFrame:
    """Resample a PnL column to 1-h buckets and compute cumulative sum."""
    if df.empty or pnl_col not in df.columns:
        return pd.DataFrame(columns=["trade_dt", "net_pnl_usdc", "cum_net_pnl_usdc"])
    hourly = (
        df.assign(trade_dt=df[dt_col].dt.floor("1h"))
        .groupby("trade_dt", as_index=False)[pnl_col]
        .sum()
        .rename(columns={pnl_col: "net_pnl_usdc"})
        .sort_values("trade_dt")
        .reset_index(drop=True)
    )
    hourly["cum_net_pnl_usdc"] = hourly["net_pnl_usdc"].cumsum()
    return hourly


def with_zero_anchor(hourly: pd.DataFrame) -> pd.DataFrame:
    """Prepend a zero anchor one hour before the first bucket."""
    if hourly.empty:
        return hourly
    anchor = pd.DataFrame({
        "trade_dt": [hourly["trade_dt"].min() - pd.Timedelta(hours=1)],
        "net_pnl_usdc": [0.0],
        "cum_net_pnl_usdc": [0.0],
    })
    return pd.concat([anchor, hourly], ignore_index=True)


# Wallet cohort PnL: from trigger rows (their actual trade_pnl)
wallet_hourly = resample_hourly_series(triggers, "dt", "wallet_trade_pnl")

# Copy-strategy PnL: from fill rows
copy_hourly = resample_hourly_series(fills, "dt", "fill_pnl_usdc")

print(f"Wallet cohort hourly buckets : {len(wallet_hourly)}")
print(f"Copy strategy hourly buckets : {len(copy_hourly)}")

Wallet cohort hourly buckets : 1094
Copy strategy hourly buckets : 966


In [16]:
wallet_hourly.head()

,trade_dt,net_pnl_usdc,cum_net_pnl_usdc
0,2026-02-16 00:00:00+00:00,-18686.864481,-18686.864481
1,2026-02-16 01:00:00+00:00,-1802.810123,-20489.674603
2,2026-02-16 02:00:00+00:00,-458.574618,-20948.249221
3,2026-02-16 03:00:00+00:00,-2064.801631,-23013.050852
4,2026-02-16 04:00:00+00:00,-1674.210525,-24687.261377


## Cumulative PnL chart

In [17]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = go.Figure()

# ── Wallet cohort trace ───────────────────────────────────────────────────────
if not wallet_hourly.empty:
    wh = with_zero_anchor(wallet_hourly)
    fig.add_trace(go.Scatter(
        x=wh["trade_dt"],
        y=wh["cum_net_pnl_usdc"],
        mode="lines",
        line=dict(dash="dot", width=2),
        opacity=0.7,
        name="Watched wallets (raw PnL)",
        hovertemplate=(
            "<b>Watched wallets</b><br>"
            "%{x|%Y-%m-%d %H:%M}<br>"
            "cum PnL: %{y:.2f} USDC<extra></extra>"
        ),
    ))

# ── Copy-strategy trace ───────────────────────────────────────────────────────
if not copy_hourly.empty:
    ch = with_zero_anchor(copy_hourly)
    fig.add_trace(go.Scatter(
        x=ch["trade_dt"],
        y=ch["cum_net_pnl_usdc"],
        mode="lines",
        line=dict(dash="solid", width=2),
        name="Copy strategy (filled)",
        hovertemplate=(
            "<b>Copy strategy</b><br>"
            "%{x|%Y-%m-%d %H:%M}<br>"
            "cum PnL: %{y:.2f} USDC<extra></extra>"
        ),
    ))

fig.add_hline(y=0, line_dash="dash", line_color="grey", line_width=1, opacity=0.5)

fig.update_layout(
    template="plotly_white",
    height=480,
    title=dict(
        text=(
            f"Copy-trade backtest — cumulative PnL (1h) | "
            f"{period_start} → {period_end} | "
            f"{len(WATCHED_WALLETS)} wallets | "
            f"worse_price={WORSE_PRICE_OFFSET:.2f} | "
            f"horizon={int(FILL_HORIZON_SECONDS)}s"
        ),
        font=dict(size=13),
    ),
    xaxis_title="Time (1h buckets)",
    yaxis_title="Cumulative net PnL (USDC)",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)

fig.show()

## Diagnostics

In [18]:
# Fill rate by side (order-level)
if not triggers.empty:
    trig_by_side = triggers.groupby("side").size().rename("triggers")
    fill_by_side = fills.groupby("side")["order_id"].nunique().rename("orders_filled")
    fill_rows_by_side = fills.groupby("side").size().rename("fill_rows")
    side_summary = pd.concat([trig_by_side, fill_by_side, fill_rows_by_side], axis=1).fillna(0).astype(int)
    side_summary["fill_rate"] = (side_summary["orders_filled"] / side_summary["triggers"] * 100).round(1)
    display(side_summary)


,triggers,orders_filled,fill_rows,fill_rate
side,,,,
BUY,179835,6179,8315,3.4
SELL,7547,4522,5527,59.9


In [19]:
# Fill source breakdown by side
if not fills.empty:
    display(
        fills.groupby(["side", "fill_source"]).size()
        .rename("count")
        .reset_index()
        .sort_values(["side", "fill_source"])
    )

,side,fill_source,count
0,BUY,opposite_token,1731
1,BUY,same_token,6584
2,SELL,opposite_token,1650
3,SELL,same_token,3877


In [20]:
df = event_ledger.assign(
    is_trigger=event_ledger["row_type"].eq("trigger"),
    is_fill=event_ledger["row_type"].eq("fill"),
)

# --- wallet-level stats (row-based) ---
wallet_stats = (
    df.groupby("wallet")
    .agg(
        total_triggers=("is_trigger", "sum"),
        total_fills=("is_fill", "sum"),
        total_fill_pnl=("fill_pnl_usdc", "sum"),
        total_trade_pnl=("wallet_trade_pnl", "sum"),
        total_pnl=("fill_pnl_usdc", "sum"),
    )
)

# --- order-level logic (trigger -> fill) ---
order_stats = (
    df.groupby(["wallet", "order_id"])
    .agg(
        has_trigger=("is_trigger", "any"),
        has_fill=("is_fill", "any"),
    )
    .assign(trigger_with_fill=lambda x: x["has_trigger"] & x["has_fill"])
    .groupby("wallet")
    .agg(triggers_with_fill=("trigger_with_fill", "sum"))
)

# --- combine ---
result = wallet_stats.join(order_stats)

# --- derived metric ---
result["fill_ratio"] = (
    result["triggers_with_fill"] / result["total_triggers"]
).fillna(0)

result.sort_values("total_pnl", ascending=False).head()

,total_triggers,total_fills,total_fill_pnl,total_trade_pnl,total_pnl,triggers_with_fill,fill_ratio
wallet,,,,,,,
0x6bc74c392c320cfe10d5be61db978a58c8444ad4,597,64,73.989707,33277.787614,73.989707,49,0.082077
0x41583f2efc720b8e2682750fffb67f2806fece9f,667,208,65.441114,8784.000339,65.441114,150,0.224888
0x8f42ae0a01c0383c7ca8bd060b86a645ee74b88f,855,151,46.412618,203043.807940,46.412618,109,0.127485
0xdaa6a2cd4ba545befb3dbdc25d2b444c46873e62,1169,443,44.462795,3363.864399,44.462795,226,0.193328
0x0cb10c40b0776e9ee8cef970af85724654dda76c,1039,109,41.878309,14411.726168,41.878309,84,0.080847


In [21]:
len(df)

201224

In [22]:
# get diff between trigger and fill timestamp per order_id
# without lambda, using groupby + agg + merge
trigger_df = df[df['row_type'] == 'trigger'].groupby('order_id')['dt'].min().reset_index()
fill_df = df[df['row_type'] == 'fill'].groupby('order_id')['dt'].min().reset_index()

merged_df = pd.merge(trigger_df, fill_df, on='order_id', how='outer', suffixes=('_trigger', '_fill'))
merged_df['diff_dt'] = merged_df['dt_fill'] - merged_df['dt_trigger']

In [23]:
merged_df[merged_df['diff_dt'].notnull()]['diff_dt'].mean().total_seconds()

60.608915

In [24]:
result[result['total_fill_pnl'] < 0].sort_values("total_fill_pnl").index.tolist()

['0x12d6cccfc7470a3f4bafc53599a4779cbf2cf2a8',
 '0x808db301f0b237020d1d877a743d6bf5f1cdebfd',
 '0x520b4a0452005517964ee864985f2026d0dc100a',
 '0x9ccc0ca985cd638f106d5f355cc2e4d11d4ce7e8',
 '0x373a949d617e60cbb25ca6df3f68018d573bf4c1',
 '0xa9e8faf20424f3efbf12abaea0e7069d7546c443',
 '0x404c04ab8622f35f6dadef2af854af1cc97c74dc',
 '0x0e3226217752d7247c67b870b72b99ba3e20535b',
 '0xa0bca9bdd8540da95060ed1fafb78aa03835d428',
 '0x0c97325a81ad05c6b58ae30d12eaaec1b8451b68',
 '0x59432fa8141b0ddf25f64c9fc90cdb5672402ef2',
 '0xec85247a3e1968e1d6ab516e3e468bddf06a8ba8',
 '0xc25427ea224b8f9fa2df801233f944006ed33f73',
 '0x0a793254e040027c5d1ca0c2f87cda8e22228e05',
 '0x4d0e9b029700163625c551a0073636f3dc5ec45b',
 '0xe0f1e7751df4035f78a5eb2f4296c98b4d5bc4bd',
 '0x50b242e0b9ac75359464c5b3be62ad6b42d8c94a',
 '0x154794795d978c5890b3f69264311f0bd966d066',
 '0x8190816855b676a5efd8c5b344135a72337bb247',
 '0xec06668bca4430ef0772ec8276c42250563a9178',
 '0x333e8f4fb0353ef31cf1a7d00789e1319381369b',
 '0xee324b845

In [25]:
# Per-wallet PnL breakdown
if not fills.empty:
    wallet_pnl_df = (
        fills.groupby("wallet")
        .agg(
            filled_orders=("order_id", "nunique"),
            fill_rows=("order_id", "count"),
            net_pnl_usdc=("fill_pnl_usdc", "sum"),
            total_qty=("fill_qty", "sum"),
            win_rate=("token_winner", lambda s: fills.loc[s.index].groupby("order_id")["token_winner"].first().mean()),
        )
        .assign(win_rate=lambda d: (d["win_rate"] * 100).round(1))
        .sort_values("net_pnl_usdc", ascending=False)
        .reset_index()
    )
    display(wallet_pnl_df)


,wallet,filled_orders,fill_rows,net_pnl_usdc,total_qty,win_rate
0,0x6bc74c392c320cfe10d5be61db978a58c8444ad4,49,64,73.989707,417.460000,69.4
1,0x41583f2efc720b8e2682750fffb67f2806fece9f,150,208,65.441114,1135.176145,42.0
2,0x8f42ae0a01c0383c7ca8bd060b86a645ee74b88f,109,151,46.412618,838.419685,68.8
3,0xdaa6a2cd4ba545befb3dbdc25d2b444c46873e62,226,443,44.462795,1366.237266,53.5
4,0x0cb10c40b0776e9ee8cef970af85724654dda76c,84,109,41.878309,645.586826,53.6
...,...,...,...,...,...,...
162,0x373a949d617e60cbb25ca6df3f68018d573bf4c1,19,24,-17.748292,140.300951,42.1
163,0x9ccc0ca985cd638f106d5f355cc2e4d11d4ce7e8,13,20,-20.232137,93.076922,7.7
164,0x520b4a0452005517964ee864985f2026d0dc100a,23,25,-31.656795,160.000000,34.8
165,0x808db301f0b237020d1d877a743d6bf5f1cdebfd,21,24,-32.690816,129.764895,57.1


In [26]:
wallet_pnl_df.head(10)['wallet'].tolist()

['0x6bc74c392c320cfe10d5be61db978a58c8444ad4',
 '0x41583f2efc720b8e2682750fffb67f2806fece9f',
 '0x8f42ae0a01c0383c7ca8bd060b86a645ee74b88f',
 '0xdaa6a2cd4ba545befb3dbdc25d2b444c46873e62',
 '0x0cb10c40b0776e9ee8cef970af85724654dda76c',
 '0x22dbd51e1a4e10fff072fa24300238c64033190f',
 '0x24c8cf69a0e0a17eee21f69d29752bfa32e823e1',
 '0x97d37d16d1774785197bfa23ffed625a8e493f3c',
 '0xf658449199d0bcf544de0ff928a3b66685f3dcfe',
 '0xc34a4af2e3a818fdcdfaa7b5b140eca017ae6007']

In [27]:
# Orders that were NOT filled at all
filled_order_ids = set(fills["order_id"].unique()) if not fills.empty else set()
unfilled_triggers = triggers[~triggers["order_id"].isin(filled_order_ids)]
print(f"Unfilled orders    : {len(unfilled_triggers):,} ({len(unfilled_triggers)/max(len(triggers),1)*100:.1f}% of all triggers)")

# Partially filled orders (filled but qty_filled < qty_ordered)
if not fills.empty:
    filled_trig = triggers[triggers["order_id"].isin(filled_order_ids)].copy()
    partial_mask = filled_trig["qty_filled"] < filled_trig["qty_ordered"] - 1e-9
    partial_fills = filled_trig[partial_mask]
    print(f"Partially filled   : {len(partial_fills):,} ({len(partial_fills)/max(len(filled_trig),1)*100:.1f}% of filled orders)")
    print(f"Fully filled       : {len(filled_trig)-len(partial_fills):,}")

if not unfilled_triggers.empty:
    print("\nUnfilled breakdown by side:")
    display(unfilled_triggers.groupby("side").size().rename("count").reset_index())


Unfilled orders    : 176,681 (94.3% of all triggers)
Partially filled   : 5,267 (49.2% of filled orders)
Fully filled       : 5,434

Unfilled breakdown by side:


,side,count
0,BUY,173656
1,SELL,3025


In [28]:
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)
fills[fills['order_id'] == 34]

,row_type,order_id,wallet,condition_id,outcome,side,dt,tx_hash,price,order_price,qty_ordered,qty_filled,fill_qty,fill_source,token_winner,exec_price,fill_pnl_usdc,wallet_trade_pnl,filled_position,shard


In [29]:
fills.groupby("wallet")["fill_pnl_usdc"].sum().sort_values(ascending=False, key=abs).head(100)

wallet
0x12d6cccfc7470a3f4bafc53599a4779cbf2cf2a8   -93.974654
0x6bc74c392c320cfe10d5be61db978a58c8444ad4    73.989707
0x41583f2efc720b8e2682750fffb67f2806fece9f    65.441114
0x8f42ae0a01c0383c7ca8bd060b86a645ee74b88f    46.412618
0xdaa6a2cd4ba545befb3dbdc25d2b444c46873e62    44.462795
0x0cb10c40b0776e9ee8cef970af85724654dda76c    41.878309
0x22dbd51e1a4e10fff072fa24300238c64033190f    38.763306
0x24c8cf69a0e0a17eee21f69d29752bfa32e823e1    35.966324
0x808db301f0b237020d1d877a743d6bf5f1cdebfd   -32.690816
0x97d37d16d1774785197bfa23ffed625a8e493f3c    32.186249
0x520b4a0452005517964ee864985f2026d0dc100a   -31.656795
0xf658449199d0bcf544de0ff928a3b66685f3dcfe    30.748039
0xc34a4af2e3a818fdcdfaa7b5b140eca017ae6007    29.137129
0xaa99a6c279d34492683db6ff1818aa1f8485dbbf    28.653059
0xca52446eaad30e546f344849ae1afd92eb360cdf    28.041178
0x7c3db723f1d4d8cb9c550095203b686cb11e5c6b    27.425949
0x8597ca63e722d6216bfc3057591fdc67ec49daee    26.256674
0xfbff8d110ff1a34e486d791b389ea4d5f8cc583

In [30]:
trigger_wallets = triggers[["order_id", "wallet"]].set_index("order_id")["wallet"]

fills_with_wallet = fills.merge(trigger_wallets, on="order_id", how="left", suffixes=("", "_trigger"))

fills_with_wallet['notional'] = fills_with_wallet['fill_qty'] * fills_with_wallet['exec_price']

fills_with_wallet.groupby(["wallet", "condition_id"]).agg(
    pnl=("fill_pnl_usdc", "sum"),
    orders_filled=("order_id", "nunique")
    ).sort_values(['wallet', "pnl"], ascending=[True, False]).head(1)

,,pnl,orders_filled
wallet,condition_id,,
0x000d257d2dc7616feaef4ae0f14600fdf50a758e,0x753b97e8082c848fb70086eba99dc2c98341f18d72c8ee4918395dbdb39390df,3.274307,1


In [31]:
fills_with_wallet.head()

,row_type,order_id,wallet,condition_id,outcome,side,dt,tx_hash,price,order_price,...,fill_qty,fill_source,token_winner,exec_price,fill_pnl_usdc,wallet_trade_pnl,filled_position,shard,wallet_trigger,notional
0,fill,1,0xd6eef76188c7068719ac881460b7431e1ae137a2,0xf86191c4b660e0f0f4669eae5638474277a3bb31e5c34e6118e5eb19e00a11cd,No,BUY,2026-02-16 00:00:51+00:00,0x0b8f56c045cc516e71896d5cf7f99a83674e137e5192bf14e8830f4eea47287c,0.96,NaN,...,0.004241,same_token,True,0.961,0.000161,NaN,0.004241,f,0xd6eef76188c7068719ac881460b7431e1ae137a2,0.004076
1,fill,3,0xd6eef76188c7068719ac881460b7431e1ae137a2,0xbebb1c8c6996bdd0783db8891f4cb764afe301c7393ae19cad4c81b0d9e25e11,Yes,BUY,2026-02-16 00:01:41+00:00,0xf2f6a76c3c0c2a1d79d01aaf557e648c61ec3c988d28d0efa9a3922a1a5b5a71,0.91,NaN,...,2.170000,same_token,True,0.910,0.193325,NaN,2.170000,b,0xd6eef76188c7068719ac881460b7431e1ae137a2,1.974700
2,fill,4,0xd6eef76188c7068719ac881460b7431e1ae137a2,0xbebb1c8c6996bdd0783db8891f4cb764afe301c7393ae19cad4c81b0d9e25e11,Yes,BUY,2026-02-16 00:02:09+00:00,0xca8ef7824274e6e5e97d02f83f95d06d1985eacac3ba1c908032330c2bb35e83,0.91,NaN,...,2.140000,same_token,True,0.910,0.190653,NaN,4.310000,b,0xd6eef76188c7068719ac881460b7431e1ae137a2,1.947400
3,fill,7,0x6bc74c392c320cfe10d5be61db978a58c8444ad4,0x28f1e9218bcd41ced05faf3c03f026721d87712556bf2828337551f03cae0a01,Yes,BUY,2026-02-16 00:04:21+00:00,0x9f45fe3c50e0382dde8e9312c4bfa8db250199513781b78595c76f16b821a9d9,0.32,NaN,...,1.470587,same_token,True,0.330,0.984808,NaN,1.470587,2,0x6bc74c392c320cfe10d5be61db978a58c8444ad4,0.485294
4,fill,7,0x6bc74c392c320cfe10d5be61db978a58c8444ad4,0x28f1e9218bcd41ced05faf3c03f026721d87712556bf2828337551f03cae0a01,Yes,BUY,2026-02-16 00:07:15+00:00,0xbb59cddeb68ff65b9e4e9e99662dbd1be6a9eec33129e8a60ca631588f62120c,0.32,NaN,...,8.529413,opposite_token,True,0.330,5.711892,NaN,10.000000,2,0x6bc74c392c320cfe10d5be61db978a58c8444ad4,2.814706


In [32]:
wallet_pnls = (
    fills_with_wallet
    .groupby(["wallet", "condition_id"])
    .agg(
        pnl=("fill_pnl_usdc", "sum"),
        orders_filled=("order_id", "nunique"),
        notional=("notional", "sum"),
    )
    .groupby("wallet")
    .agg(
        pnl=("pnl", "sum"),
        notional=("notional", "sum"),
        unique_conditions=("pnl", "size"),  # ✅ count rows
        total_orders_filled=("orders_filled", "sum"),
    )
    .sort_values("pnl", ascending=False)
)

In [33]:
wallet_pnls.head()

,pnl,notional,unique_conditions,total_orders_filled
wallet,,,,
0x6bc74c392c320cfe10d5be61db978a58c8444ad4,73.989707,169.689368,25,49
0x41583f2efc720b8e2682750fffb67f2806fece9f,65.441114,521.638585,42,150
0x8f42ae0a01c0383c7ca8bd060b86a645ee74b88f,46.412618,460.442863,29,109
0xdaa6a2cd4ba545befb3dbdc25d2b444c46873e62,44.462795,656.320756,54,226
0x0cb10c40b0776e9ee8cef970af85724654dda76c,41.878309,259.139548,34,84


In [34]:
MAX_WALLET_EXPOSURE = 1000.0  # USDC
wallet_pnls['pnl_limited'] = np.where(
    wallet_pnls['notional'] <= MAX_WALLET_EXPOSURE,
    wallet_pnls['pnl'],
    wallet_pnls['pnl'] * MAX_WALLET_EXPOSURE / wallet_pnls['notional']
)

In [35]:
wallet_pnls

,pnl,notional,unique_conditions,total_orders_filled,pnl_limited
wallet,,,,,
0x6bc74c392c320cfe10d5be61db978a58c8444ad4,73.989707,169.689368,25,49,73.989707
0x41583f2efc720b8e2682750fffb67f2806fece9f,65.441114,521.638585,42,150,65.441114
0x8f42ae0a01c0383c7ca8bd060b86a645ee74b88f,46.412618,460.442863,29,109,46.412618
0xdaa6a2cd4ba545befb3dbdc25d2b444c46873e62,44.462795,656.320756,54,226,44.462795
0x0cb10c40b0776e9ee8cef970af85724654dda76c,41.878309,259.139548,34,84,41.878309
...,...,...,...,...,...
0x373a949d617e60cbb25ca6df3f68018d573bf4c1,-17.748292,84.939353,14,19,-17.748292
0x9ccc0ca985cd638f106d5f355cc2e4d11d4ce7e8,-20.232137,22.121747,12,13,-20.232137
0x520b4a0452005517964ee864985f2026d0dc100a,-31.656795,95.561233,18,23,-31.656795


In [36]:
result = (
    fills_with_wallet
    # 1️⃣ PnL per market (condition) per wallet
    .groupby(["wallet", "condition_id"])["fill_pnl_usdc"]
    .sum()
    
    # 2️⃣ Then per wallet
    .groupby("wallet")
    .agg(
        unique_conditions="count",   # number of markets traded
        median_market_pnl="median",  # median PnL across markets
    )
    .sort_values("unique_conditions", ascending=False)
    .head(20)
)

In [37]:
result.sort_values("median_market_pnl", ascending=False).head(100)

,unique_conditions,median_market_pnl
wallet,,
0x41583f2efc720b8e2682750fffb67f2806fece9f,42,0.818047
0x9ba43501360dcacaca09caa523401c7447d8f8c2,37,0.515525
0xdaa6a2cd4ba545befb3dbdc25d2b444c46873e62,54,0.381002
0x6560abb717a8e5f3afaacd1a31e9c03eed71e164,34,0.367875
0x41cb8653fd75ebf2a46741d224dc596e5a72a5df,31,0.226284
0xa9e8faf20424f3efbf12abaea0e7069d7546c443,35,0.205637
0x97d37d16d1774785197bfa23ffed625a8e493f3c,42,0.110534
0xd9755ad7dec7773eea6714e9921d4fa29bee5e9d,31,0.100110
0x68c24bf4a8ad4d79a6fe4b8eec6f93a02dfd1711,35,0.097298


In [38]:
fills_with_wallet.head()

,row_type,order_id,wallet,condition_id,outcome,side,dt,tx_hash,price,order_price,...,fill_qty,fill_source,token_winner,exec_price,fill_pnl_usdc,wallet_trade_pnl,filled_position,shard,wallet_trigger,notional
0,fill,1,0xd6eef76188c7068719ac881460b7431e1ae137a2,0xf86191c4b660e0f0f4669eae5638474277a3bb31e5c34e6118e5eb19e00a11cd,No,BUY,2026-02-16 00:00:51+00:00,0x0b8f56c045cc516e71896d5cf7f99a83674e137e5192bf14e8830f4eea47287c,0.96,NaN,...,0.004241,same_token,True,0.961,0.000161,NaN,0.004241,f,0xd6eef76188c7068719ac881460b7431e1ae137a2,0.004076
1,fill,3,0xd6eef76188c7068719ac881460b7431e1ae137a2,0xbebb1c8c6996bdd0783db8891f4cb764afe301c7393ae19cad4c81b0d9e25e11,Yes,BUY,2026-02-16 00:01:41+00:00,0xf2f6a76c3c0c2a1d79d01aaf557e648c61ec3c988d28d0efa9a3922a1a5b5a71,0.91,NaN,...,2.170000,same_token,True,0.910,0.193325,NaN,2.170000,b,0xd6eef76188c7068719ac881460b7431e1ae137a2,1.974700
2,fill,4,0xd6eef76188c7068719ac881460b7431e1ae137a2,0xbebb1c8c6996bdd0783db8891f4cb764afe301c7393ae19cad4c81b0d9e25e11,Yes,BUY,2026-02-16 00:02:09+00:00,0xca8ef7824274e6e5e97d02f83f95d06d1985eacac3ba1c908032330c2bb35e83,0.91,NaN,...,2.140000,same_token,True,0.910,0.190653,NaN,4.310000,b,0xd6eef76188c7068719ac881460b7431e1ae137a2,1.947400
3,fill,7,0x6bc74c392c320cfe10d5be61db978a58c8444ad4,0x28f1e9218bcd41ced05faf3c03f026721d87712556bf2828337551f03cae0a01,Yes,BUY,2026-02-16 00:04:21+00:00,0x9f45fe3c50e0382dde8e9312c4bfa8db250199513781b78595c76f16b821a9d9,0.32,NaN,...,1.470587,same_token,True,0.330,0.984808,NaN,1.470587,2,0x6bc74c392c320cfe10d5be61db978a58c8444ad4,0.485294
4,fill,7,0x6bc74c392c320cfe10d5be61db978a58c8444ad4,0x28f1e9218bcd41ced05faf3c03f026721d87712556bf2828337551f03cae0a01,Yes,BUY,2026-02-16 00:07:15+00:00,0xbb59cddeb68ff65b9e4e9e99662dbd1be6a9eec33129e8a60ca631588f62120c,0.32,NaN,...,8.529413,opposite_token,True,0.330,5.711892,NaN,10.000000,2,0x6bc74c392c320cfe10d5be61db978a58c8444ad4,2.814706


In [39]:
from polymarket_analysis.data.data_catalogue import load_markets_processed
mdf = load_markets_processed()
mdf = mdf[
    ~(mdf['primary_tag'].isin(['Sports', 'Crypto']))
    & (mdf['winner_token_id'].notna())
    ].copy().reset_index(drop=True)


In [40]:
mdf.head()

,condition_id,end_date_iso,question,tags,primary_tag,winner_token_id,outcome
0,0x055176c9ebd8775c281ad540d5c16b3323e316507aef2d45c84f061cbc6bbcdc,2022-08-21T00:00:00Z,"MLB: Who will win Boston Red Sox v. Baltimore Orioles, scheduled for August 21, 7:10 PM ET?",[All],All,9612890763764062692282935414227141810568206972440321500296202304471805951204,Orioles
1,0x1409177eae7f1b0406bc0eea9d2715c9f76d88ec7765aed03cf39d59f36008f7,2023-01-06T00:00:00Z,Will a House Speaker be elected by end of Thursday?,[All],All,56419231299380534710656783098291368308638365867243518354348835265264811381897,No
2,0xe7d21325e5cfccbdaddf6ab6ef9bf477cf3aa6615dd36c8c408da99a1dd83237,2023-01-14T00:00:00Z,Will Volodymyr Zelenskyy be the 2023 TIME Person of the Year?,"[Politics, Joe Biden, time magazine, person of the year, Awards, technology, news, future predic...",Politics,5775128592397269721112080310923554806977502333079078864830955469822139503940,No
3,0xdb4b110117e21b9f4505b969dac4790b883315c5782d437eb27e256deaa88d7a,2023-01-14T00:00:00Z,Will Elon Musk be the 2023 TIME Person of the Year?,"[Politics, Joe Biden, time magazine, person of the year, Awards, technology, news, future predic...",Politics,111925628487742883146996804476220726156559363451635893056065831198346085566170,No
4,0xd6165c1a30269d0799b27ee6ea90c048dfd6b27276cbd2aa7e18fe1c4562faa8,2023-01-14T00:00:00Z,Will Benjamin Netanyahu be the 2023 TIME Person of the Year?,"[Politics, Joe Biden, time magazine, person of the year, Awards, technology, news, future predic...",Politics,115040502429502340199300991115762950152511906206476760034830683863926486402417,No


## Browser plots: PnL and positions

Interactive Plotly charts for cumulative PnL and signed positions. Market hovers include `end_date_iso` from `mdf`.


In [41]:
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.renderers.default = "browser"

TOP_WALLETS_PNL = 10
TOP_MARKETS_PNL = 10
TOP_WALLETS_POSITION = 10
TOP_MARKETS_POSITION = 10

def _short_wallet(wallet: str) -> str:
    wallet = str(wallet)
    return f"{wallet[:6]}...{wallet[-4:]}" if len(wallet) > 14 else wallet

def _short_text(text: str, width: int = 72) -> str:
    if pd.isna(text):
        return "Unknown market"
    text = " ".join(str(text).split())
    return text if len(text) <= width else text[: width - 3] + "..."

def _build_total_ts(df: pd.DataFrame, value_col: str, net_col: str, cum_col: str) -> pd.DataFrame:
    total = (
        df.sort_values("dt")
        .groupby("dt", as_index=False)[value_col]
        .sum()
        .rename(columns={value_col: net_col})
    )
    total[cum_col] = total[net_col].cumsum()
    return total

def _build_position_snapshots(df: pd.DataFrame) -> pd.DataFrame:
    series_cols = [
        "wallet",
        "wallet_label",
        "condition_id",
        "market_label",
        "end_date_iso",
        "market_close_dt",
        "outcome",
    ]

    active = df[
        df["market_close_dt"].isna() | (df["dt"] < df["market_close_dt"])
    ].copy()

    snapshots = (
        active.sort_values(series_cols + ["dt", "order_id"])
        .groupby(series_cols + ["dt"], as_index=False, dropna=False)
        .agg(
            token_position=("filled_position", "last"),
            exec_price=("exec_price", "last"),
        )
    )
    snapshots["usdc_position"] = snapshots["token_position"] * snapshots["exec_price"]

    if snapshots.empty:
        snapshots["token_delta"] = pd.Series(dtype=float)
        snapshots["usdc_delta"] = pd.Series(dtype=float)
        return snapshots

    reset = (
        snapshots.groupby(series_cols, as_index=False, dropna=False)
        .agg(
            token_position=("token_position", "last"),
            usdc_position=("usdc_position", "last"),
        )
    )
    reset = reset[
        reset["market_close_dt"].notna()
        & ((reset["token_position"].abs() > 1e-12) | (reset["usdc_position"].abs() > 1e-12))
    ].copy()
    reset["dt"] = reset["market_close_dt"]
    reset["token_position"] = 0.0
    reset["usdc_position"] = 0.0

    combined = pd.concat(
        [
            snapshots[series_cols + ["dt", "token_position", "usdc_position"]],
            reset[series_cols + ["dt", "token_position", "usdc_position"]],
        ],
        ignore_index=True,
    )
    combined = (
        combined.sort_values(series_cols + ["dt"])
        .drop_duplicates(subset=series_cols + ["dt"], keep="last")
        .reset_index(drop=True)
    )
    combined["token_delta"] = combined.groupby(series_cols, dropna=False)["token_position"].diff()
    combined["usdc_delta"] = combined.groupby(series_cols, dropna=False)["usdc_position"].diff()
    combined["token_delta"] = combined["token_delta"].fillna(combined["token_position"])
    combined["usdc_delta"] = combined["usdc_delta"].fillna(combined["usdc_position"])
    return combined

market_meta = (
    mdf[["condition_id", "question", "end_date_iso", "primary_tag"]]
    .drop_duplicates(subset=["condition_id"])
    .copy()
)
market_meta["end_date"] = pd.to_datetime(market_meta["end_date_iso"], utc=True, errors="coerce")
market_meta["market_close_dt"] = market_meta["end_date"].dt.floor("D") + pd.Timedelta(days=1)

plot_fills = fills.copy()
plot_fills["dt"] = pd.to_datetime(plot_fills["dt"], utc=True)
plot_fills = plot_fills.merge(market_meta, on="condition_id", how="left")
plot_fills["wallet_label"] = plot_fills["wallet"].map(_short_wallet)
plot_fills["market_label"] = plot_fills["question"].map(_short_text)
plot_fills["market_label"] = np.where(
    plot_fills["market_label"].eq("Unknown market"),
    plot_fills["condition_id"].str[:12],
    plot_fills["market_label"],
)

pnl_total_ts = _build_total_ts(plot_fills, "fill_pnl_usdc", "net_pnl_usdc", "cum_pnl_usdc")

wallet_pnl_totals = (
    plot_fills.groupby(["wallet", "wallet_label"], as_index=False, dropna=False)
    .agg(total_pnl_usdc=("fill_pnl_usdc", "sum"))
)
top_wallets_pnl = wallet_pnl_totals.reindex(
    wallet_pnl_totals["total_pnl_usdc"].abs().sort_values(ascending=False).index
).head(TOP_WALLETS_PNL).copy()
wallet_pnl_ts = (
    plot_fills[plot_fills["wallet"].isin(top_wallets_pnl["wallet"])]
    .groupby(["wallet", "wallet_label", "dt"], as_index=False, dropna=False)
    .agg(net_pnl_usdc=("fill_pnl_usdc", "sum"))
    .sort_values(["wallet", "dt"])
)
wallet_pnl_ts["cum_pnl_usdc"] = wallet_pnl_ts.groupby("wallet", dropna=False)["net_pnl_usdc"].cumsum()

market_pnl_totals = (
    plot_fills.groupby(["condition_id", "market_label", "end_date_iso"], as_index=False, dropna=False)
    .agg(total_pnl_usdc=("fill_pnl_usdc", "sum"))
)
top_markets_pnl = market_pnl_totals.reindex(
    market_pnl_totals["total_pnl_usdc"].abs().sort_values(ascending=False).index
).head(TOP_MARKETS_PNL).copy()
market_pnl_ts = (
    plot_fills[plot_fills["condition_id"].isin(top_markets_pnl["condition_id"])]
    .groupby(["condition_id", "market_label", "end_date_iso", "dt"], as_index=False, dropna=False)
    .agg(net_pnl_usdc=("fill_pnl_usdc", "sum"))
    .sort_values(["condition_id", "dt"])
)
market_pnl_ts["cum_pnl_usdc"] = market_pnl_ts.groupby("condition_id", dropna=False)["net_pnl_usdc"].cumsum()

granular_position_ts = _build_position_snapshots(plot_fills)

wallet_position_all = (
    granular_position_ts.groupby(["wallet", "wallet_label", "dt"], as_index=False, dropna=False)
    .agg(
        net_token_position=("token_delta", "sum"),
        net_usdc_position=("usdc_delta", "sum"),
    )
    .sort_values(["wallet", "dt"])
)
wallet_position_all["cum_token_position"] = wallet_position_all.groupby("wallet", dropna=False)["net_token_position"].cumsum()
wallet_position_all["cum_usdc_position"] = wallet_position_all.groupby("wallet", dropna=False)["net_usdc_position"].cumsum()
wallet_position_rank = (
    wallet_position_all.groupby(["wallet", "wallet_label"], as_index=False, dropna=False)
    .agg(max_abs_usdc_position=("cum_usdc_position", lambda s: s.abs().max()))
)
top_wallets_position = wallet_position_rank.sort_values("max_abs_usdc_position", ascending=False).head(TOP_WALLETS_POSITION).copy()
wallet_position_ts = wallet_position_all[wallet_position_all["wallet"].isin(top_wallets_position["wallet"])].copy()

market_position_all = (
    granular_position_ts.groupby(["condition_id", "market_label", "end_date_iso", "dt"], as_index=False, dropna=False)
    .agg(
        net_token_position=("token_delta", "sum"),
        net_usdc_position=("usdc_delta", "sum"),
    )
    .sort_values(["condition_id", "dt"])
)
market_position_all["cum_token_position"] = market_position_all.groupby("condition_id", dropna=False)["net_token_position"].cumsum()
market_position_all["cum_usdc_position"] = market_position_all.groupby("condition_id", dropna=False)["net_usdc_position"].cumsum()
market_position_rank = (
    market_position_all.groupby(["condition_id", "market_label", "end_date_iso"], as_index=False, dropna=False)
    .agg(max_abs_usdc_position=("cum_usdc_position", lambda s: s.abs().max()))
)
top_markets_position = market_position_rank.sort_values("max_abs_usdc_position", ascending=False).head(TOP_MARKETS_POSITION).copy()
market_position_ts = market_position_all[market_position_all["condition_id"].isin(top_markets_position["condition_id"])].copy()

token_total_ts = _build_total_ts(granular_position_ts, "token_delta", "net_token_position", "cum_token_position")
usdc_total_ts = _build_total_ts(granular_position_ts, "usdc_delta", "net_usdc_position", "cum_usdc_position")

print(
    f"Prepared plot datasets: total pnl={len(pnl_total_ts)}, "
    f"wallet pnl series={wallet_pnl_ts['wallet'].nunique()}, "
    f"market pnl series={market_pnl_ts['condition_id'].nunique()}, "
    f"wallet position series={wallet_position_ts['wallet'].nunique()}, "
    f"market position series={market_position_ts['condition_id'].nunique()}"
)


Prepared plot datasets: total pnl=10740, wallet pnl series=10, market pnl series=10, wallet position series=10, market position series=10


In [42]:
fig_pnl = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=(
        "Total cumulative PnL",
        f"Per-wallet cumulative PnL (top {TOP_WALLETS_PNL} by abs total PnL)",
        f"Per-market cumulative PnL (top {TOP_MARKETS_PNL} by abs total PnL)",
    ),
)

fig_pnl.add_trace(
    go.Scatter(
        x=pnl_total_ts["dt"],
        y=pnl_total_ts["cum_pnl_usdc"],
        mode="lines",
        name="Total",
        line=dict(width=3, color="#1f77b4"),
        hovertemplate="%{x|%Y-%m-%d %H:%M}<br>Total cum PnL: %{y:,.2f} USDC<extra></extra>",
    ),
    row=1,
    col=1,
)

wallet_totals_map = top_wallets_pnl.set_index("wallet")["total_pnl_usdc"].to_dict()
for wallet in top_wallets_pnl["wallet"]:
    sub = wallet_pnl_ts[wallet_pnl_ts["wallet"] == wallet]
    label = sub["wallet_label"].iloc[0]
    total = wallet_totals_map[wallet]
    fig_pnl.add_trace(
        go.Scatter(
            x=sub["dt"],
            y=sub["cum_pnl_usdc"],
            mode="lines",
            name=f"{label} ({total:,.1f})",
            hovertemplate=(
                f"Wallet: {wallet}<br>"
                + "%{x|%Y-%m-%d %H:%M}<br>"
                + "Cum PnL: %{y:,.2f} USDC<extra></extra>"
            ),
        ),
        row=2,
        col=1,
    )

market_totals_map = top_markets_pnl.set_index("condition_id")["total_pnl_usdc"].to_dict()
for condition_id in top_markets_pnl["condition_id"]:
    sub = market_pnl_ts[market_pnl_ts["condition_id"] == condition_id]
    label = sub["market_label"].iloc[0]
    end_date_iso = sub["end_date_iso"].iloc[0]
    total = market_totals_map[condition_id]
    fig_pnl.add_trace(
        go.Scatter(
            x=sub["dt"],
            y=sub["cum_pnl_usdc"],
            mode="lines",
            name=f"{label} ({total:,.1f})",
            hovertemplate=(
                f"Market: {label}<br>"
                + f"end_date_iso: {end_date_iso}<br>"
                + f"condition_id: {condition_id}<br>"
                + "%{x|%Y-%m-%d %H:%M}<br>"
                + "Cum PnL: %{y:,.2f} USDC<extra></extra>"
            ),
        ),
        row=3,
        col=1,
    )

for row in (1, 2, 3):
    fig_pnl.add_hline(y=0, line_dash="dash", line_color="grey", line_width=1, opacity=0.5, row=row, col=1)

fig_pnl.update_yaxes(title_text="USDC", row=1, col=1)
fig_pnl.update_yaxes(title_text="USDC", row=2, col=1)
fig_pnl.update_yaxes(title_text="USDC", row=3, col=1)
fig_pnl.update_xaxes(title_text="Time", row=3, col=1)
fig_pnl.update_layout(
    template="plotly_white",
    height=1200,
    title="Stage 2 copy-trade backtest - cumulative PnL",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)
fig_pnl.show(renderer="browser")


In [43]:
fig_token_pos = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=(
        "Total signed token position",
        f"Per-wallet signed token position (top {TOP_WALLETS_POSITION} by max abs USDC position)",
        f"Per-market signed token position (top {TOP_MARKETS_POSITION} by max abs USDC position)",
    ),
)

fig_token_pos.add_trace(
    go.Scatter(
        x=token_total_ts["dt"],
        y=token_total_ts["cum_token_position"],
        mode="lines",
        line_shape="hv",
        name="Total token position",
        line=dict(width=3, color="#2ca02c"),
        hovertemplate="%{x|%Y-%m-%d %H:%M}<br>Signed token position: %{y:,.2f}<extra></extra>",
    ),
    row=1,
    col=1,
)

for wallet in top_wallets_position["wallet"]:
    sub = wallet_position_ts[wallet_position_ts["wallet"] == wallet]
    label = sub["wallet_label"].iloc[0]
    max_abs = sub["cum_usdc_position"].abs().max()
    fig_token_pos.add_trace(
        go.Scatter(
            x=sub["dt"],
            y=sub["cum_token_position"],
            mode="lines",
            line_shape="hv",
            name=f"{label} ({max_abs:,.1f} USDC)",
            hovertemplate=(
                f"Wallet: {wallet}<br>"
                + "%{x|%Y-%m-%d %H:%M}<br>"
                + "Signed token position: %{y:,.2f}<extra></extra>"
            ),
        ),
        row=2,
        col=1,
    )

for condition_id in top_markets_position["condition_id"]:
    sub = market_position_ts[market_position_ts["condition_id"] == condition_id]
    label = sub["market_label"].iloc[0]
    end_date_iso = sub["end_date_iso"].iloc[0]
    max_abs = sub["cum_usdc_position"].abs().max()
    fig_token_pos.add_trace(
        go.Scatter(
            x=sub["dt"],
            y=sub["cum_token_position"],
            mode="lines",
            line_shape="hv",
            name=f"{label} ({max_abs:,.1f} USDC)",
            hovertemplate=(
                f"Market: {label}<br>"
                + f"end_date_iso: {end_date_iso}<br>"
                + f"condition_id: {condition_id}<br>"
                + "%{x|%Y-%m-%d %H:%M}<br>"
                + "Signed token position: %{y:,.2f}<extra></extra>"
            ),
        ),
        row=3,
        col=1,
    )

for row in (1, 2, 3):
    fig_token_pos.add_hline(y=0, line_dash="dash", line_color="grey", line_width=1, opacity=0.5, row=row, col=1)

fig_token_pos.update_yaxes(title_text="Tokens", row=1, col=1)
fig_token_pos.update_yaxes(title_text="Tokens", row=2, col=1)
fig_token_pos.update_yaxes(title_text="Tokens", row=3, col=1)
fig_token_pos.update_xaxes(title_text="Time", row=3, col=1)
fig_token_pos.update_layout(
    template="plotly_white",
    height=1200,
    title="Stage 2 copy-trade backtest - signed token positions",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)
fig_token_pos.show(renderer="browser")


In [44]:
fig_usdc_pos = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=(
        "Total signed USDC position (using traded price)",
        f"Per-wallet signed USDC position (top {TOP_WALLETS_POSITION} by max abs USDC position)",
        f"Per-market signed USDC position (top {TOP_MARKETS_POSITION} by max abs USDC position)",
    ),
)

fig_usdc_pos.add_trace(
    go.Scatter(
        x=usdc_total_ts["dt"],
        y=usdc_total_ts["cum_usdc_position"],
        mode="lines",
        line_shape="hv",
        name="Total USDC position",
        line=dict(width=3, color="#d62728"),
        hovertemplate="%{x|%Y-%m-%d %H:%M}<br>Signed USDC position: %{y:,.2f}<extra></extra>",
    ),
    row=1,
    col=1,
)

for wallet in top_wallets_position["wallet"]:
    sub = wallet_position_ts[wallet_position_ts["wallet"] == wallet]
    label = sub["wallet_label"].iloc[0]
    max_abs = sub["cum_usdc_position"].abs().max()
    fig_usdc_pos.add_trace(
        go.Scatter(
            x=sub["dt"],
            y=sub["cum_usdc_position"],
            mode="lines",
            line_shape="hv",
            name=f"{label} ({max_abs:,.1f})",
            hovertemplate=(
                f"Wallet: {wallet}<br>"
                + "%{x|%Y-%m-%d %H:%M}<br>"
                + "Signed USDC position: %{y:,.2f}<extra></extra>"
            ),
        ),
        row=2,
        col=1,
    )

for condition_id in top_markets_position["condition_id"]:
    sub = market_position_ts[market_position_ts["condition_id"] == condition_id]
    label = sub["market_label"].iloc[0]
    end_date_iso = sub["end_date_iso"].iloc[0]
    max_abs = sub["cum_usdc_position"].abs().max()
    fig_usdc_pos.add_trace(
        go.Scatter(
            x=sub["dt"],
            y=sub["cum_usdc_position"],
            mode="lines",
            line_shape="hv",
            name=f"{label} ({max_abs:,.1f})",
            hovertemplate=(
                f"Market: {label}<br>"
                + f"end_date_iso: {end_date_iso}<br>"
                + f"condition_id: {condition_id}<br>"
                + "%{x|%Y-%m-%d %H:%M}<br>"
                + "Signed USDC position: %{y:,.2f}<extra></extra>"
            ),
        ),
        row=3,
        col=1,
    )

for row in (1, 2, 3):
    fig_usdc_pos.add_hline(y=0, line_dash="dash", line_color="grey", line_width=1, opacity=0.5, row=row, col=1)

fig_usdc_pos.update_yaxes(title_text="USDC", row=1, col=1)
fig_usdc_pos.update_yaxes(title_text="USDC", row=2, col=1)
fig_usdc_pos.update_yaxes(title_text="USDC", row=3, col=1)
fig_usdc_pos.update_xaxes(title_text="Time", row=3, col=1)
fig_usdc_pos.update_layout(
    template="plotly_white",
    height=1200,
    title="Stage 2 copy-trade backtest - signed USDC positions",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
)
fig_usdc_pos.show(renderer="browser")
